### Code Hist.

 - CODE  
    &ensp; : Model - KIER Method 02(Clustering)

  - DATE      &ensp; 2023-03-05 Created  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1) Dataset : KIER / TCN    
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 2) Model : LightGBM  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 3)   

 - Related Link  
    &ensp; : 

# 01. Code

## 01-01. Init

### 01-01-01. Init_Module Import

In [ ]:
#region Basic_Import
## Basic
import os
os.path.dirname(os.path.abspath('__file__'))
import sys
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

import math
import random

## Datetime
import time
import datetime as dt
from datetime import datetime, date, timedelta

import glob
from glob import glob
import requests
import json

## 시각화
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.model_selection import train_test_split

# CLustering 알고리즘의 성능 평가 측도
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import metrics

import urllib
from urllib.request import urlopen
from urllib.parse import urlencode, unquote, quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

## Init.
pd.options.display.float_format = '{:.10f}'.format
#endregion Basic_Import

In [ ]:
## Import_DL
str_tar = "tf"
## For Torch
if str_tar == "torch":
    import torch
    import torch.nn as nn
    from torch.nn.utils import weight_norm
    print("Torch Imported")
## For TF
elif str_tar == "tf":
    import tensorflow as tf
    import tensorflow_addons as tfa
    print("Tensorflow Imported")
else:
    print("Error : Cannot be used except for Keywords")
    print(" : torch / tf")

In [ ]:
print(tf.__version__)

In [ ]:
# !pip install catboost

## LGBM
from catboost import Pool, CatBoostRegressor

# !pip install lightgbm

## LGBM
import lightgbm as lgbm
from lightgbm import LGBMRegressor

## XGBoost
import xgboost as xgb
from xgboost import plot_importance, plot_tree, XGBClassifier

## TCN
# !pip install keras-tcn
from tcn import TCN, tcn_full_summary, compiled_tcn
from glob import glob
from tqdm import tqdm
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.preprocessing import MinMaxScaler
from IPython.display import clear_output

In [ ]:
## Import_Local
from core import data_datetime as com_date
from core import provider_kasi as com_Holi
from core import data_analysis as com_Analysis
from core import data_preprocessing as com_Prep
from core import data_visualization as com_Visual
from core import provider_kma as com_ASOS
from core import provider_kdhc as com_KDHC
from core import provider_kier as com_KIER

### 01-01-02. Config (Directory, Params)

In [ ]:
## Init_config
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = "1"
random.seed(SEED)

In [ ]:
## Define Todate str
str_now_ymd = pd.datetime.now().date()
str_now_y = pd.datetime.now().year
str_now_m = pd.datetime.now().month
str_now_d = pd.datetime.now().day
str_now_hr = pd.datetime.now().hour
str_now_min = pd.datetime.now().minute

print(pd.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

In [ ]:
## Dict_Domain
dict_domain = {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"} ## GAS는 사용하지 않음.
int_domain = 0
str_domain = str(dict_domain[int_domain])

dict_col_accu = {0 : "ACTUAL_ACCU_EFF" ## ELEC
                 , 1 : "ACCU_HEAT" ## HEAT
                 , 2 : "ACCU_FLOW" ## WATER
                 , 3 : "ACCU_HEAT" ## HOT 열량
                 , 4 : "ACCU_FLOW" ## HOT 유량
                 , 99 : "ACCU_FLOW" ## GAS
                 }
str_col_accu = str(str_domain + "_" + str(dict_col_accu[int_domain]))

dict_col_inst = {0 : "INST_EFF" ## ELEC_ACCU/INST_EFF
                , 1 : "INST_HEAT" ## HEAT_ACCU/INST_HEAT
                , 2 : "INST_FLOW" ## WATER_ACCU/INST_FLOW
                , 3 : "INST_HEAT" ## HOT_ACCU/INST_HEAT
                , 4 : "INST_FLOW" ## HOT_ACCU/INST_FLOW
                , 99 : "INST_FLOW" ## GAS_ACCU/INST_FLOW
                } 
str_col_inst = str(str_domain + "_" + str(dict_col_inst[int_domain]))

## Directory Root
str_dirData = "../data/data_Energy_KIER/"
str_dir_raw = '../data/data_Energy_KIER/KIER_0_Raw/'
str_dirName_bld = '../data/data_Energy_KIER/KIER_1_BLD/'
str_dirName_f = '../data/data_Energy_KIER/KIER_2_F_' + str_domain + '/'
str_dirName_h = '../data/data_Energy_KIER/KIER_3_H_' + str_domain + '/'

## File
str_fileRaw = str('KIER_RAW_' + str_domain + '_2023-11-12.csv')
str_fileRaw_hList = str('KIER_hList_' + str_domain + '.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

## 01-02. Data Load (df_raw)

### 01-02-01. KDHC Heat Usage (Intergrated)

In [ ]:
## KMA_ASOS Data
str_dir_kmaAsos = "../data/data_KMA_ASOS/"

## Interpolate / Filled ASOS Data
str_file = 'ASOS_119_2010-2024_HR_INTP.csv'
Data_ASOS = pd.read_csv(str_dir_kmaAsos + str_file
                        , index_col = 0)
Data_ASOS['METER_DATE'] = pd.to_datetime(Data_ASOS['METER_DATE'])
Data_ASOS

In [ ]:
## Cluster 기준 Interval
list_interval = ['10MIN', '1H', '1DAY', '1WEEK', '1MONTH']
str_interval = list_interval[4]
str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
print(str_interval)
print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
df_kier_h_cluster

In [ ]:
list_kier_h_all = df_kier_h_cluster['h_index']
print(len(list_kier_h_all))
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))
list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
print(len(list_kier_h_c2))

In [ ]:
## 사용량 Data Load
## 1시간 단위
str_file = 'KIER_' + str_domain + '_INST_1H_InstBaseUpdated.csv'
df_raw = pd.read_csv(str_dirName_h + str_file
                     , index_col = 0)
df_raw

In [ ]:
## 전체 사용량 합계
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_all]
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
## 0으로 표기되는 마지막행 제거
df_kier_h_all = df_kier_h_all[:-1]
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
df_kier_h_all.dropna()

In [ ]:
## Cluster별 사용량 합계
## ■ C00
df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c0]
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
## 0으로 표기되는 마지막행 제거
df_kier_h_c0 = df_kier_h_c0[:-1]
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
df_kier_h_c0.dropna()

## ■ C01
df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c1]
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
## 0으로 표기되는 마지막행 제거
df_kier_h_c1 = df_kier_h_c1[:-1]
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
df_kier_h_c1.dropna()

## ■ C02
df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c2]
df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
## 0으로 표기되는 마지막행 제거
df_kier_h_c2 = df_kier_h_c2[:-1]
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
df_kier_h_c2.dropna()

In [ ]:
# 에너지 사용량 총계만
# df_kier_h_all = df_kier_h_all[['METER_DATE', str_domain + '_INST_SUM_ALL']].dropna()

# df_kier_h_c0 = df_kier_h_c0[['METER_DATE', str_domain + '_INST_SUM_C0']].dropna()

# df_kier_h_c1 = df_kier_h_c1[['METER_DATE', str_domain + '_INST_SUM_C1']].dropna()

# df_kier_h_c2 = df_kier_h_c2[['METER_DATE', str_domain + '_INST_SUM_C2']].dropna()


In [ ]:
## 날씨 데이터 추가
# df_kier_h_all = pd.merge(df_kier_h_all, Data_ASOS
#                          , how = 'left', on = ['METER_DATE'])
# df_kier_h_c0 = pd.merge(df_kier_h_c0, Data_ASOS
#                         , how = 'left', on = ['METER_DATE'])
# df_kier_h_c1 = pd.merge(df_kier_h_c1, Data_ASOS
#                         , how = 'left', on = ['METER_DATE'])
# df_kier_h_c2 = pd.merge(df_kier_h_c2, Data_ASOS
#                         , how = 'left', on = ['METER_DATE'])

In [ ]:
print(df_kier_h_all.shape)
print(df_kier_h_all.columns)
print(df_kier_h_c0.shape)
print(df_kier_h_c0.columns)
print(df_kier_h_c1.shape)
print(df_kier_h_c1.columns)
print(df_kier_h_c2.shape)
print(df_kier_h_c2.columns)

In [ ]:
# df_dt = pd.DataFrame()
# df_dt['METER_DATE'] = pd.to_datetime(df_int_ALL['METER_DATE'])

# df_int_ALL = df_int_ALL[df_int_ALL.columns[18:].to_list()]
# df_int_ALL['METER_DATE'] = pd.to_datetime(df_dt['METER_DATE'])
# df_int_C0 = df_int_C0[df_int_C0.columns[18:].to_list()]
# df_int_C0['METER_DATE'] = pd.to_datetime(df_dt['METER_DATE'])
# df_int_C1 = df_int_C1[df_int_C1.columns[18:].to_list()]
# df_int_C1['METER_DATE'] = pd.to_datetime(df_dt['METER_DATE'])
# df_int_C2 = df_int_C2[df_int_C2.columns[18:].to_list()]
# df_int_C2['METER_DATE'] = pd.to_datetime(df_dt['METER_DATE'])
# df_int_ALL

## 01-06. Data Split (Train/Test Setting)

In [ ]:
## 모든 세대
df_raw = df_kier_h_all
str_col_tar = str_domain + '_INST_SUM_ALL'
## 모든 세대
# df_raw = df_kier_h_c0
# str_col_tar = str_domain + '_INST_SUM_C0'
## 모든 세대
# df_raw = df_kier_h_c1
# str_col_tar = str_domain + '_INST_SUM_C1'
## 모든 세대
# df_raw = df_kier_h_c2
# str_col_tar = str_domain + '_INST_SUM_C2'

# df_raw = com_date.create_col_ymdhm(df_raw, 'METER_DATE')
# df_raw = com_date.create_col_weekdays(df_raw, 'METER_DATE')

df_raw = df_raw.drop(columns = ['METER_DATE']).dropna()
# df_raw = df_raw.drop(columns = ['METER_DATE', 'day_of_the_week']).dropna()
# df_raw = df_raw.drop(columns = ['METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'None']).dropna()

trainSet_Origin, testSet_Origin = train_test_split(df_raw, test_size=0.3, shuffle=False)
print(trainSet_Origin.shape, testSet_Origin.shape)

In [ ]:
df_raw['ELEC_INST_SUM_ALL'].mean()

In [ ]:
df_raw.isna().sum()

In [ ]:
trainSet = trainSet_Origin
testSet = testSet_Origin

## Input / Target Split
trainXX = trainSet.drop([str_col_tar],axis=1)
trainYY = trainSet[[str_col_tar]]
testXX = testSet.drop([str_col_tar],axis=1)
testYY = testSet[[str_col_tar]]

In [ ]:
trainXXindex = trainXX.index
trainXXcolumns = trainXX.columns

trainYYindex = trainYY.index
trainYYcolumns = trainYY.columns

testXXindex = testXX.index
testXXcolumns = testXX.columns

testYYindex = testYY.index
testYYcolumns = testYY.columns

In [ ]:
d_trainXX=pd.DataFrame(trainXX, index=trainXXindex, columns=trainXXcolumns)
d_trainYY=trainYY

d_testXX=pd.DataFrame(testXX, index=testXXindex, columns=testXXcolumns)
d_testYY=testYY

## 01-07. Model

### 01-07-01. DL  
1) TCN
2) 
3) 
4) 
5) 
6) 

In [ ]:
## Init Param_TCN
TIME_STEP = 1
# NUM_FEATRUE = 25
NUM_FEATRUE = 348

In [ ]:
def buildDataSet(traindata, testdata, TIME_STEP):
    xdata = []
    ydata = []

    for i in range(len(traindata) - TIME_STEP + 1):
        tx = traindata.iloc[i : i + TIME_STEP]
        ty = testdata.iloc[i + TIME_STEP - 1]
        xdata.append(tx)
        ydata.append(ty)

    return np.array(xdata), np.array(ydata)

trainX, trainY = buildDataSet(trainXX, trainYY, TIME_STEP)
testX, testY = buildDataSet(testXX, testYY, TIME_STEP)

In [ ]:
## Train/Test Check
print(trainX.shape, trainY.shape)
print(testX.shape, testY.shape)

In [ ]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")

def build_model():
    inputs = tf.keras.Input(shape=(TIME_STEP, NUM_FEATRUE))    
#     inputs = tf.keras.Input(shape=(0, NUM_FEATRUE))    
    x = TCN(nb_filters=64
            , kernel_size=3
            , nb_stacks=2
            , dilations=(1, 2, 4, 8, 16, 32, 64)
            , padding='causal'
            , use_skip_connections=True
            , dropout_rate=0.
            , return_sequences=False
            , kernel_initializer='he_normal'
            , use_batch_norm=False
            , use_layer_norm=False
            , use_weight_norm=False
            , activation="tanh"
#             , activation="relu"
           )(inputs)
    outputs = tf.keras.layers.Dense(4, dtype=tf.float32)(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = build_model()
model.summary()

#### 2-1. Train Model_TCN

In [ ]:
import time
tm_start = time.time()

In [ ]:
class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch % 10 == 0:
            clear_output(wait=True)

# 대회용 custom metric 생성, 단 참조만으로 활용
@tf.function
def rmse(y_true, y_pred):
    return tf.math.sqrt(tf.math.reduce_mean(tf.math.square(tf.math.subtract(y_true, y_pred))))

@tf.function
def rmse_r2(y_true, y_pred):
    _rmse = rmse(y_true, y_pred)

    residual = tf.math.reduce_sum(tf.math.square(tf.math.subtract(y_true, y_pred)))
    total = tf.math.reduce_sum(tf.math.square(tf.math.subtract(y_true, tf.math.reduce_mean(y_true))))
    r2 = tf.math.subtract(1.0, tf.math.divide(residual, total))

    rmse_r2 = tf.math.divide(_rmse, r2)    
    return tf.where(rmse_r2 < 0., 999., rmse_r2)

callbacks = [
                DisplayCallback(),
                tf.keras.callbacks.EarlyStopping(monitor="loss", mode="min", verbose=0, patience=5),
                tf.keras.callbacks.ModelCheckpoint("../models/V2.h5", monitor="rmse_r2", mode="min", save_best_only=True)
            ]

def scale_fn(x):
    return 1/(2.**(x-1))

lr = tfa.optimizers.CyclicalLearningRate(1e-4, 1e-2, step_size=20, scale_fn=scale_fn, scale_mode="cycle")
optimizer = tfa.optimizers.Lookahead(tfa.optimizers.AdaBelief(learning_rate=lr))
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(optimizer)
model.compile(optimizer=optimizer, loss="mse", metrics=[rmse_r2])
history = model.fit(trainX, trainY, epochs=10, callbacks=callbacks)    

In [ ]:
tm_code = time.time() - tm_start
print("time : ", tm_code)

In [ ]:
## Print Model Summary
print(model.summary())

In [ ]:
# ## Save Model and Training Hist
# if args.name is not None:
#     with open(modelsDir + "{}.history".format(args.name), "wb") as file_pi:
#         h = history.history
#         h["epoch_time"] = custom_callback.times
#         h["epoch_lr"] = custom_callback.learningRates
#         pickle.dump(h, file_pi)
#     model.save(modelsDir + "{}.h5".format(args.name))

In [ ]:
d_trainXX.info()

In [ ]:
pred_test = model.predict(testX)

plt.plot(testY, color = 'red')
plt.plot(pred_test, color = 'blue')

In [ ]:
model_pred = pred_test
model_preds = np.reshape(model_pred,(-1,1))

d_actual = testY#.to_numpy()
d_actuals = np.reshape(d_actual,(-1,1))

In [ ]:
## Code Time
print('TIME : ', tm_code)

com_Analysis.model_sk_metrics(d_actual, model_pred)

In [ ]:
print(str_domain)
print(str_interval)
print(str_col_tar)